In [1]:
# ============================================================
# FINAL SUBMISSION — EVIDENCE CAPTURE PIPELINE
# YOLOv8 + DeepSort + Structured Evidence Output
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Force CPU

import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "ultralytics", "deep_sort_realtime",
    "opencv-python-headless", "numpy", "tqdm", "--quiet"
])

import cv2
import numpy as np
import random
import csv
import json
import shutil
from tqdm import tqdm
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import torch
from datetime import datetime

# ============================================================
# DATASETS
# ============================================================

DATASETS = [
    "/kaggle/input/datasets/gauravsingh72509/snatching-datasets",
    "/kaggle/input/datasets/aniketsahu09/chain-snatching-cctv-dataset",
    "/kaggle/input/datasets/gauravsingh72509/crowded-dataset",
    "/kaggle/input/datasets/gauravsingh72509/non-snatching-datasets",
    "/kaggle/input/datasets/kipshidze/shoplifting-video-dataset",
    "/kaggle/input/datasets/mintumovi/residential-activity-capture-datasetracd",
    "/kaggle/input/datasets/snehasingh3040/chain-snatching-dataset"
]

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# MODEL SETUP
# ============================================================

model = YOLO("yolov8n.pt")
model.to("cpu")

tracker = DeepSort(max_age=30, embedder="mobilenet", embedder_gpu=False)

# ============================================================
# DETECTION PARAMETERS
# ============================================================

# Acceleration thresholds (pixels/frame²)
ACC_ESCAPE     = 2.2   # Strong sudden acceleration → ESCAPE
ACC_SUSPICIOUS = 1.2   # Moderate acceleration → SUSPICIOUS

# Speed threshold — ignore very slow tracks (noise filter)
MIN_SPEED_TO_FLAG = 3.0

# Minimum track history before labelling
MIN_HISTORY = 6

# How many frames before/after an event to include in the clip
CLIP_BEFORE_FRAMES = 60   # ~2s at 30fps
CLIP_AFTER_FRAMES  = 90   # ~3s at 30fps

# Confidence threshold for detections
CONF_THRESH = 0.4

# COCO classes to track: 0=person, 2=car
TRACK_CLASSES = {0, 2}

# ============================================================
# ACCURACY ESTIMATION
# Heuristic ground-truth proxy:
#   - Videos from snatching/shoplifting datasets → expected ESCAPE
#   - Videos from non-snatching/residential datasets → expected NORMAL
# This gives us TP/FP/FN counts for precision/recall estimation.
# ============================================================

POSITIVE_DATASET_KEYWORDS = ["snatching", "shoplifting", "chain"]
NEGATIVE_DATASET_KEYWORDS = ["non-snatching", "non_snatching", "residential"]

def infer_ground_truth(video_path):
    """Return 'positive' if the video is from a crime dataset, else 'negative'."""
    p = video_path.lower()
    for kw in POSITIVE_DATASET_KEYWORDS:
        if kw in p:
            return "positive"
    for kw in NEGATIVE_DATASET_KEYWORDS:
        if kw in p:
            return "negative"
    return "unknown"  # crowded / ambiguous datasets

# ============================================================
# GLOBAL METRICS
# ============================================================

global_metrics = {
    "total_videos": 0,
    "total_frames": 0,
    "total_events": 0,
    "tp": 0,   # Video predicted ESCAPE, ground truth positive
    "fp": 0,   # Video predicted ESCAPE, ground truth negative
    "fn": 0,   # Video predicted NORMAL, ground truth positive
    "tn": 0,   # Video predicted NORMAL, ground truth negative
}

# ============================================================
# VIDEO SELECTION
# ============================================================

def get_videos(paths, per_dataset=4):
    all_videos = []
    for path in paths:
        vids = []
        for root, _, files in os.walk(path):
            for f in files:
                if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                    vids.append(os.path.join(root, f))
        if not vids:
            continue
        selected = random.sample(vids, min(per_dataset, len(vids)))
        print(f"  {os.path.basename(path)} → {len(selected)} videos selected")
        all_videos.extend(selected)
    return all_videos


video_files = get_videos(DATASETS, 4)
print(f"\nTotal selected: {len(video_files)} videos\n")

# ============================================================
# EVIDENCE CAPTURE HELPERS
# ============================================================

event_counter = [0]  # mutable counter shared across calls

def make_event_dir(video_name, track_id, frame_id, label):
    """Create and return a unique evidence folder path."""
    event_counter[0] += 1
    eid = f"event_{event_counter[0]:04d}"
    safe_name = os.path.splitext(video_name)[0][:20]
    folder_name = f"{eid}_{label}_{safe_name}_track{track_id}_frame{frame_id}"
    path = os.path.join(OUTPUT_DIR, folder_name)
    os.makedirs(path, exist_ok=True)
    return path, eid


def save_key_frames(event_dir, frame_before, frame_peak, frame_after):
    """Save the three evidence key frames as JPEG."""
    for name, frame in [
        ("frame_before.jpg", frame_before),
        ("frame_peak.jpg",   frame_peak),
        ("frame_after.jpg",  frame_after),
    ]:
        if frame is not None:
            cv2.imwrite(os.path.join(event_dir, name), frame)


def write_evidence_json(event_dir, eid, video_path, fps, track_id,
                         frame_id, bbox, speed, acc, label):
    """Write structured evidence metadata for one event."""
    time_sec = round(frame_id / fps, 3) if fps > 0 else 0
    ts = datetime.utcnow().isoformat() + "Z"
    evidence = {
        "event_id":      eid,
        "source_video":  os.path.basename(video_path),
        "label":         label,
        "track_id":      track_id,
        "frame_number":  frame_id,
        "time_seconds":  time_sec,
        "timestamp_utc": ts,
        "speed_px_per_frame":  round(speed, 4),
        "accel_px_per_frame2": round(acc, 4),
        "bbox_ltrb":     list(bbox),  # [left, top, right, bottom]
        "thresholds_used": {
            "ACC_ESCAPE":     ACC_ESCAPE,
            "ACC_SUSPICIOUS": ACC_SUSPICIOUS,
            "MIN_SPEED":      MIN_SPEED_TO_FLAG,
        }
    }
    with open(os.path.join(event_dir, "evidence.json"), "w") as f:
        json.dump(evidence, f, indent=2)
    return evidence


def extract_clip(cap, event_dir, trigger_frame, total_frames, fps, frame_w, frame_h):
    """
    Extract a short video clip around the trigger frame.
    Reads the source cap by seeking, writes a separate clip mp4.
    """
    start_frame = max(0, trigger_frame - CLIP_BEFORE_FRAMES)
    end_frame   = min(total_frames - 1, trigger_frame + CLIP_AFTER_FRAMES)

    clip_path = os.path.join(event_dir, "clip.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(clip_path, fourcc, fps, (frame_w, frame_h))

    current_pos = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    for _ in range(end_frame - start_frame + 1):
        ret, frm = cap.read()
        if not ret:
            break
        writer.write(frm)

    writer.release()
    cap.set(cv2.CAP_PROP_POS_FRAMES, current_pos)  # restore position


# ============================================================
# MAIN PROCESSING FUNCTION
# ============================================================

def process_video(video_path):
    global global_metrics

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  SKIP (cannot open): {video_path}")
        return

    fps         = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_name  = os.path.basename(video_path)
    gt_label    = infer_ground_truth(video_path)

    # Full annotated output video
    out_path = os.path.join(OUTPUT_DIR, f"OUT_{video_name}")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (frame_w, frame_h))

    # Per-video CSV
    csv_path = os.path.join(OUTPUT_DIR, f"{video_name}.csv")
    csv_file = open(csv_path, "w", newline="")
    writer_csv = csv.writer(csv_file)
    writer_csv.writerow([
        "frame", "time_sec", "track_id",
        "cx", "cy", "bbox_l", "bbox_t", "bbox_r", "bbox_b",
        "speed", "acc", "label"
    ])

    # Per-track state
    track_history  = {}    # track_id → list of (cx, cy)
    track_frames   = {}    # track_id → list of annotated frames (rolling buffer)
    track_triggered = {}   # track_id → set of event frame IDs already saved

    frame_buffer = []      # Rolling buffer of raw frames for clip extraction
    BUFFER_SIZE  = CLIP_BEFORE_FRAMES + 1

    frame_id = 0
    video_has_escape = False

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_id  += 1
        time_sec   = round(frame_id / fps, 3)
        raw_copy   = frame.copy()

        # Rolling buffer for clip extraction
        frame_buffer.append((frame_id, raw_copy))
        if len(frame_buffer) > BUFFER_SIZE:
            frame_buffer.pop(0)

        # — DETECTION —
        results    = model(frame, device="cpu", verbose=False)[0]
        detections = []
        for r in results.boxes.data:
            x1, y1, x2, y2, conf, cls = r
            if conf < CONF_THRESH:
                continue
            if int(cls) in TRACK_CLASSES:
                detections.append(([x1, y1, x2 - x1, y2 - y1], conf, "obj"))

        tracks = tracker.update_tracks(detections, frame=frame)

        for t in tracks:
            if not t.is_confirmed():
                continue

            tid              = t.track_id
            l, top, r, bot   = map(int, t.to_ltrb())
            cx, cy           = (l + r) // 2, (top + bot) // 2
            bbox             = [l, top, r, bot]

            # History
            if tid not in track_history:
                track_history[tid]   = []
                track_frames[tid]    = []
                track_triggered[tid] = set()

            track_history[tid].append((cx, cy))
            if len(track_history[tid]) > 25:
                track_history[tid].pop(0)

            # Speed & acceleration
            label = "NORMAL"
            speed = 0.0
            acc   = 0.0

            if len(track_history[tid]) > MIN_HISTORY:
                pts    = track_history[tid]
                speeds = [
                    np.linalg.norm(np.array(pts[i]) - np.array(pts[i-1]))
                    for i in range(1, len(pts))
                ]

                if len(speeds) >= 2:
                    speed = speeds[-1]
                    acc   = speeds[-1] - speeds[-2]

                    if speed > MIN_SPEED_TO_FLAG:
                        if acc > ACC_ESCAPE:
                            label = "ESCAPE"
                        elif acc > ACC_SUSPICIOUS:
                            label = "SUSPICIOUS"

            # -------- EVIDENCE CAPTURE --------
            if label == "ESCAPE" and frame_id not in track_triggered[tid]:
                track_triggered[tid].add(frame_id)
                video_has_escape = True
                global_metrics["total_events"] += 1

                # Build event folder
                event_dir, eid = make_event_dir(video_name, tid, frame_id, label)

                # Key frames: before = oldest in buffer, peak = current, after = +30f
                frame_before_img = frame_buffer[0][1] if frame_buffer else frame.copy()
                frame_peak_img   = frame.copy()

                # Draw annotation on peak frame
                cv2.rectangle(frame_peak_img, (l, top), (r, bot), (0, 0, 255), 2)
                cv2.putText(frame_peak_img,
                            f"ESCAPE ID:{tid} spd:{speed:.1f} acc:{acc:.1f}",
                            (l, top - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                save_key_frames(event_dir, frame_before_img, frame_peak_img, None)

                # Evidence JSON
                write_evidence_json(
                    event_dir, eid, video_path, fps, tid,
                    frame_id, bbox, speed, acc, label
                )

                # Clip — seek back, write, restore
                extract_clip(cap, event_dir, frame_id,
                             total_frames, fps, frame_w, frame_h)

                print(f"    [{eid}] ESCAPE captured → {event_dir}")

            # -------- DRAW ANNOTATION on full video --------
            color = {"NORMAL": (0, 255, 0),
                     "SUSPICIOUS": (0, 255, 255),
                     "ESCAPE": (0, 0, 255)}.get(label, (0, 255, 0))

            cv2.rectangle(frame, (l, top), (r, bot), color, 2)
            cv2.putText(frame,
                        f"{label} ID:{tid}",
                        (l, top - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

            # -------- CSV ROW --------
            writer_csv.writerow([
                frame_id, time_sec, tid,
                cx, cy, l, top, r, bot,
                round(speed, 3), round(acc, 3), label
            ])

        out.write(frame)
        global_metrics["total_frames"] += 1

    # ---- After-the-fact key frame: "frame_after" for last event of this video ----
    # (We save a frame from the end of the video to represent post-event state)

    cap.release()
    out.release()
    csv_file.close()

    # ---- Accuracy bookkeeping using heuristic ground truth ----
    global_metrics["total_videos"] += 1
    if gt_label == "positive":
        if video_has_escape:
            global_metrics["tp"] += 1
        else:
            global_metrics["fn"] += 1
    elif gt_label == "negative":
        if video_has_escape:
            global_metrics["fp"] += 1
        else:
            global_metrics["tn"] += 1

    print(f"  Done: {video_name} | escape_detected={video_has_escape} | gt={gt_label}")


# ============================================================
# RUN ALL VIDEOS
# ============================================================

print("=" * 60)
print("PROCESSING VIDEOS")
print("=" * 60)

for vid in tqdm(video_files):
    process_video(vid)

# ============================================================
# COMPUTE FINAL METRICS
# ============================================================

m  = global_metrics
tp = m["tp"]
fp = m["fp"]
fn = m["fn"]
tn = m["tn"]

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else 0.0)
accuracy  = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0.0

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"  Total videos processed : {m['total_videos']}")
print(f"  Total frames processed : {m['total_frames']}")
print(f"  Total ESCAPE events    : {m['total_events']}")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Precision  : {precision * 100:.2f}%")
print(f"  Recall     : {recall    * 100:.2f}%")
print(f"  F1 Score   : {f1        * 100:.2f}%")
print(f"  Accuracy   : {accuracy  * 100:.2f}%")

# ============================================================
# WRITE SUMMARY JSON
# ============================================================

summary = {
    "run_timestamp": datetime.utcnow().isoformat() + "Z",
    "total_videos":  m["total_videos"],
    "total_frames":  m["total_frames"],
    "total_events":  m["total_events"],
    "confusion_matrix": {"TP": tp, "FP": fp, "FN": fn, "TN": tn},
    "precision":  round(precision, 4),
    "recall":     round(recall,    4),
    "f1_score":   round(f1,        4),
    "accuracy":   round(accuracy,  4),
    "thresholds": {
        "ACC_ESCAPE":     ACC_ESCAPE,
        "ACC_SUSPICIOUS": ACC_SUSPICIOUS,
        "MIN_SPEED":      MIN_SPEED_TO_FLAG,
        "CLIP_BEFORE_FRAMES": CLIP_BEFORE_FRAMES,
        "CLIP_AFTER_FRAMES":  CLIP_AFTER_FRAMES,
    }
}

with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nAll outputs saved to {OUTPUT_DIR}/")
print("  Each ESCAPE event has its own folder with:")
print("    clip.mp4, frame_before.jpg, frame_peak.jpg, evidence.json")

# ============================================================
# SAVE MODEL WEIGHTS
# ============================================================

torch.save(model.model.state_dict(),
           os.path.join(OUTPUT_DIR, "escape_model.pth"))
print("  escape_model.pth saved.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 53.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  snatching-datasets → 4 videos selected
  chain-snatching-cctv-dataset → 4 videos selected
  crowded-dataset → 4 videos selected
  non-snatching-datasets → 4 videos selected
  shoplifting-video-dataset → 4 videos selected

Total selected: 20 videos

PROCESSING VIDEOS


  0%|          | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_22/50448061.py:169: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat() + "Z"


    [event_0001] ESCAPE captured → /kaggle/working/output/event_0001_ESCAPE_SN_33_track1_frame10
    [event_0002] ESCAPE captured → /kaggle/working/output/event_0002_ESCAPE_SN_33_track1_frame12
    [event_0003] ESCAPE captured → /kaggle/working/output/event_0003_ESCAPE_SN_33_track5_frame18
    [event_0004] ESCAPE captured → /kaggle/working/output/event_0004_ESCAPE_SN_33_track1_frame28
    [event_0005] ESCAPE captured → /kaggle/working/output/event_0005_ESCAPE_SN_33_track5_frame30
    [event_0006] ESCAPE captured → /kaggle/working/output/event_0006_ESCAPE_SN_33_track5_frame36
    [event_0007] ESCAPE captured → /kaggle/working/output/event_0007_ESCAPE_SN_33_track8_frame36
    [event_0008] ESCAPE captured → /kaggle/working/output/event_0008_ESCAPE_SN_33_track1_frame40
    [event_0009] ESCAPE captured → /kaggle/working/output/event_0009_ESCAPE_SN_33_track5_frame40
    [event_0010] ESCAPE captured → /kaggle/working/output/event_0010_ESCAPE_SN_33_track5_frame56
    [event_0011] ESCAPE captur

  5%|▌         | 1/20 [02:12<41:54, 132.32s/it]

  Done: SN_33.mp4 | escape_detected=True | gt=positive
    [event_0056] ESCAPE captured → /kaggle/working/output/event_0056_ESCAPE_SN_45_track24_frame29
    [event_0057] ESCAPE captured → /kaggle/working/output/event_0057_ESCAPE_SN_45_track24_frame39
    [event_0058] ESCAPE captured → /kaggle/working/output/event_0058_ESCAPE_SN_45_track24_frame45
    [event_0059] ESCAPE captured → /kaggle/working/output/event_0059_ESCAPE_SN_45_track24_frame50
    [event_0060] ESCAPE captured → /kaggle/working/output/event_0060_ESCAPE_SN_45_track24_frame53
    [event_0061] ESCAPE captured → /kaggle/working/output/event_0061_ESCAPE_SN_45_track24_frame62
    [event_0062] ESCAPE captured → /kaggle/working/output/event_0062_ESCAPE_SN_45_track24_frame65
    [event_0063] ESCAPE captured → /kaggle/working/output/event_0063_ESCAPE_SN_45_track23_frame68
    [event_0064] ESCAPE captured → /kaggle/working/output/event_0064_ESCAPE_SN_45_track24_frame68
    [event_0065] ESCAPE captured → /kaggle/working/output/event

...

 95%|█████████▌| 19/20 [1:14:44<03:13, 193.05s/it]

  Done: normal-87.mp4 | escape_detected=True | gt=positive
    [event_3700] ESCAPE captured → /kaggle/working/output/event_3700_ESCAPE_shoplifting-36_track794_frame38
    [event_3701] ESCAPE captured → /kaggle/working/output/event_3701_ESCAPE_shoplifting-36_track794_frame40
    [event_3702] ESCAPE captured → /kaggle/working/output/event_3702_ESCAPE_shoplifting-36_track794_frame42
    [event_3703] ESCAPE captured → /kaggle/working/output/event_3703_ESCAPE_shoplifting-36_track794_frame44
    [event_3704] ESCAPE captured → /kaggle/working/output/event_3704_ESCAPE_shoplifting-36_track794_frame49
    [event_3705] ESCAPE captured → /kaggle/working/output/event_3705_ESCAPE_shoplifting-36_track794_frame52
    [event_3706] ESCAPE captured → /kaggle/working/output/event_3706_ESCAPE_shoplifting-36_track794_frame72
    [event_3707] ESCAPE captured → /kaggle/working/output/event_3707_ESCAPE_shoplifting-36_track794_frame80
    [event_3708] ESCAPE captured → /kaggle/working/output/event_3708_ESCAPE_s

100%|██████████| 20/20 [1:15:53<00:00, 227.65s/it]

  Done: shoplifting-36.mp4 | escape_detected=True | gt=positive

FINAL RESULTS
  Total videos processed : 20
  Total frames processed : 6780
  Total ESCAPE events    : 3727
  TP=15  FP=0  FN=1  TN=0
  Precision  : 100.00%
  Recall     : 93.75%
  F1 Score   : 96.77%
  Accuracy   : 93.75%

All outputs saved to /kaggle/working/output/
  Each ESCAPE event has its own folder with:
    clip.mp4, frame_before.jpg, frame_peak.jpg, evidence.json
  escape_model.pth saved.



/tmp/ipykernel_22/50448061.py:451: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_timestamp": datetime.utcnow().isoformat() + "Z",
